# Preprocessing — Nettoyage et Préparation des Données

Ce notebook transforme les données brutes en features prêtes pour la modélisation.
Le pipeline sklearn est entraîné ici sur le jeu d'entraînement uniquement,
puis sérialisé pour être réutilisé dans chaque notebook modèle.

**Règle fondamentale :** On ne touche pas au jeu de test avant l'évaluation finale.
Tout ce qui est appris (scaler, encodeur) est fité sur le train uniquement.

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.data.loader import (
    load_raw, make_train_test_split,
    PROCESSED_DIR, TARGET_CLASSIFICATION
)
from src.data.preprocessor import (
    add_revenue_at_risk, add_engagement_score,
    get_feature_lists, fit_and_save_pipeline,
    get_feature_names_after_encoding, PROCESSED_DIR
)

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df_raw = load_raw()
print(df_raw.shape)

Dataset chargé : 10,000 lignes, 32 colonnes
(10000, 32)


## 1. Feature engineering

In [2]:
# On ajoute le score d'engagement composite avant le split.
# Ce score sera une feature supplémentaire pour les modèles de classification.
df = add_engagement_score(df_raw)

# revenue_at_risk est la variable cible de la régression.
# Au stade du preprocessing, on utilise un proxy (taux de churn observé).
# Il sera recalculé dans le notebook 07 avec les vraies probabilités de churn.
df = add_revenue_at_risk(df)

print(f"Colonnes ajoutées : engagement_score, revenue_at_risk")
print(f"\nAperçu engagement_score :")
print(df['engagement_score'].describe().round(3))

# Vérification du score par statut churn — les churners doivent avoir un score plus bas
print(f"\nEngagement moyen non-churn : {df[df['churn']==0]['engagement_score'].mean():.3f}")
print(f"Engagement moyen churners  : {df[df['churn']==1]['engagement_score'].mean():.3f}")

Colonnes ajoutées : engagement_score, revenue_at_risk

Aperçu engagement_score :
count    10000.000
mean         0.430
std          0.091
min          0.148
25%          0.364
50%          0.431
75%          0.495
max          0.718
Name: engagement_score, dtype: float64

Engagement moyen non-churn : 0.432
Engagement moyen churners  : 0.413


## 2. Split train / test

In [3]:
# Split stratifié 80/20 sur le churn.
# La stratification est indispensable avec 10% de churn pour garantir
# une représentation équilibrée de la classe minoritaire dans les deux jeux.
X_train, X_test, y_train, y_test = make_train_test_split(df, TARGET_CLASSIFICATION)

print(f"\nDistribution churn — train : {y_train.mean():.3f} | test : {y_test.mean():.3f}")
print("La stratification est respectée : les proportions sont similaires dans les deux jeux.")

Train : 8,000 lignes | Test : 2,000 lignes
Taux de churn train : 0.102 | test : 0.102

Distribution churn — train : 0.102 | test : 0.102
La stratification est respectée : les proportions sont similaires dans les deux jeux.


## 3. Construction et sauvegarde du pipeline de prétraitement

In [4]:
num_features, cat_features = get_feature_lists(X_train)

print(f"Features numériques ({len(num_features)}) :", num_features)
print(f"\nFeatures catégorielles ({len(cat_features)}) :", cat_features)

Features numériques (19) : ['age', 'tenure_months', 'monthly_logins', 'weekly_active_days', 'avg_session_time', 'features_used', 'usage_growth_rate', 'last_login_days_ago', 'monthly_fee', 'total_revenue', 'payment_failures', 'support_tickets', 'avg_resolution_time', 'csat_score', 'escalations', 'email_open_rate', 'marketing_click_rate', 'nps_score', 'referral_count']

Features catégorielles (11) : ['gender', 'country', 'city', 'customer_segment', 'signup_channel', 'contract_type', 'payment_method', 'discount_applied', 'price_increase_last_3m', 'complaint_type', 'survey_response']


In [5]:
# On fitte le pipeline sur le train uniquement — jamais sur le test.
preprocessor = fit_and_save_pipeline(X_train, num_features, cat_features)

# Transformation des deux jeux
X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

feature_names = get_feature_names_after_encoding(preprocessor, num_features, cat_features)

print(f"\nShape après preprocessing :")
print(f"  Train : {X_train_processed.shape}")
print(f"  Test  : {X_test_processed.shape}")
print(f"  Nombre de features : {len(feature_names)}")

Pipeline sauvegardé : /mnt/d/Etude/Master cours/Projet/Data_Science/Projet_Data_Science/data/processed/preprocessing_pipeline.joblib

Shape après preprocessing :
  Train : (8000, 58)
  Test  : (2000, 58)
  Nombre de features : 58


## 4. Sauvegarde des jeux de données prétraités

In [6]:
# On sauvegarde les données sous forme de DataFrame avec les noms de colonnes.
# Cela facilite l'utilisation dans les notebooks suivants et l'interprétabilité.
import joblib

X_train_df = pd.DataFrame(X_train_processed, columns=feature_names)
X_test_df = pd.DataFrame(X_test_processed, columns=feature_names)

X_train_df.to_csv(PROCESSED_DIR / 'X_train.csv', index=False)
X_test_df.to_csv(PROCESSED_DIR / 'X_test.csv', index=False)
y_train.reset_index(drop=True).to_csv(PROCESSED_DIR / 'y_train.csv', index=False)
y_test.reset_index(drop=True).to_csv(PROCESSED_DIR / 'y_test.csv', index=False)

# On sauvegarde aussi la liste des noms de features pour les notebooks modèles
joblib.dump(feature_names, PROCESSED_DIR / 'feature_names.joblib')

# Sauvegarde des données pour la régression
from src.data.loader import make_train_test_split as mts, TARGET_REGRESSION
X_train_reg, X_test_reg, y_train_reg, y_test_reg = mts(df, TARGET_REGRESSION)
X_train_reg_proc = preprocessor.transform(X_train_reg)
X_test_reg_proc = preprocessor.transform(X_test_reg)

X_train_reg_df = pd.DataFrame(X_train_reg_proc, columns=feature_names)
X_test_reg_df = pd.DataFrame(X_test_reg_proc, columns=feature_names)
X_train_reg_df.to_csv(PROCESSED_DIR / 'X_train_regression.csv', index=False)
X_test_reg_df.to_csv(PROCESSED_DIR / 'X_test_regression.csv', index=False)
y_train_reg.reset_index(drop=True).to_csv(PROCESSED_DIR / 'y_train_regression.csv', index=False)
y_test_reg.reset_index(drop=True).to_csv(PROCESSED_DIR / 'y_test_regression.csv', index=False)

print("Fichiers sauvegardés dans ../data/processed/ :")
import os
for f in sorted(os.listdir(PROCESSED_DIR)):
    print(f'  {f}')

Train : 8,000 lignes | Test : 2,000 lignes
Fichiers sauvegardés dans data/processed/ :
  X_test.csv
  X_test_regression.csv
  X_train.csv
  X_train_regression.csv
  feature_names.joblib
  fig_categorical_churn_rates.png
  fig_churn_distribution.png
  fig_correlation_matrix.png
  fig_financial_analysis.png
  fig_numeric_distributions.png
  preprocessing_pipeline.joblib
  y_test.csv
  y_test_regression.csv
  y_train.csv
  y_train_regression.csv


## 5. Vérification de la qualité du preprocessing

In [7]:
# Vérification que les features numériques sont bien centrées/réduites
train_df_check = pd.DataFrame(X_train_processed, columns=feature_names)
num_check = train_df_check[num_features].describe().T[['mean', 'std']]
print("Vérification StandardScaler (mean ≈ 0, std ≈ 1) :")
print(num_check.round(3).to_string())

Vérification StandardScaler (mean ≈ 0, std ≈ 1) :
                      mean  std
age                    0.0  1.0
tenure_months          0.0  1.0
monthly_logins        -0.0  1.0
weekly_active_days     0.0  1.0
avg_session_time       0.0  1.0
features_used         -0.0  1.0
usage_growth_rate      0.0  1.0
last_login_days_ago   -0.0  1.0
monthly_fee           -0.0  1.0
total_revenue         -0.0  1.0
payment_failures      -0.0  1.0
support_tickets        0.0  1.0
avg_resolution_time   -0.0  1.0
csat_score             0.0  1.0
escalations           -0.0  1.0
email_open_rate       -0.0  1.0
marketing_click_rate  -0.0  1.0
nps_score             -0.0  1.0
referral_count        -0.0  1.0


In [8]:
# Vérification qu'il n'y a plus de valeurs manquantes
nans_train = pd.DataFrame(X_train_processed, columns=feature_names).isnull().sum().sum()
nans_test = pd.DataFrame(X_test_processed, columns=feature_names).isnull().sum().sum()
print(f"Valeurs manquantes après preprocessing — train : {nans_train} | test : {nans_test}")
assert nans_train == 0 and nans_test == 0, "Des NaN subsistent après le preprocessing !"

Valeurs manquantes après preprocessing — train : 0 | test : 0
